The Research Question I'm Trying To Solve Here is :
Youtube and Facebook are different. Facebook is more people based, while youtube is more content based. Based on this how do the two relate to each other? Do communities form in the same way? Which platform has the most well connected users? How can we improve and create healthy communities.

In [5]:
#Load both datasets in social_network_analysis. com-youtube.ungraph and facebook_combined.txt
import pandas as pd
import networkx as nx

facebook_path = "social_network_analysis/facebook_combined.txt"
youtube_path = "social_network_analysis/com-youtube.ungraph.txt"

# Load Facebook network (simple edge list, no header lines)
facebook_graph = nx.read_edgelist(facebook_path, nodetype=int)

# Load YouTube network (has 4 comment lines starting with #, so we skip them)
youtube_graph = nx.read_edgelist(youtube_path, nodetype=int, comments='#')



In [6]:
# Display Dataset Info
print(f"Facebook Data")
print(f"   Nodes (users)     : {nx.number_of_nodes(facebook_graph):,}")
print(f"   Edges (connections): {nx.number_of_edges(facebook_graph):,}")
print(f"YouTube Data")
print(f"   Nodes (users)     : {nx.number_of_nodes(youtube_graph):,}")
print(f"   Edges (connections): {nx.number_of_edges(youtube_graph):,}")

# Numerical Attributes?
# Dataset is a Node Graph :)
# - Degree: Number of connections each user has, basically who is connected to who
# - Density: How 'grouped' together are users.

Facebook Data
   Nodes (users)     : 4,039
   Edges (connections): 88,234
YouTube Data
   Nodes (users)     : 1,134,890
   Edges (connections): 2,987,624


In [7]:
import numpy as np

# Get the Summary Information for Each Network Graph
def get_degree_stats(graph, name):
    degrees = [deg for _, deg in graph.degree()]
    
    stats = {
        "Network"           : name,
        "Nodes"             : graph.number_of_nodes(),
        "Edges"             : graph.number_of_edges(),
        "Mean Degree"       : round(np.mean(degrees), 4),
        "Median Degree"     : round(np.median(degrees), 4),
        "Min Degree"        : int(np.min(degrees)),
        "Max Degree"        : int(np.max(degrees)),
        "Std Dev Degree"    : round(np.std(degrees), 4),
        "Density"           : round(nx.density(graph), 6),
    }
    return stats

# Calculate stats for both networks
fb_stats  = get_degree_stats(facebook_graph, "Facebook")
yt_stats  = get_degree_stats(youtube_graph,  "YouTube")

# Display as a Table
stats_df = pd.DataFrame([fb_stats, yt_stats]).set_index("Network")
print("Summary of Data")
print("")
print(stats_df.to_string())

Summary of Data

            Nodes    Edges  Mean Degree  Median Degree  Min Degree  Max Degree  Std Dev Degree   Density
Network                                                                                                 
Facebook     4039    88234       43.691           25.0           1        1045         52.4141  0.010820
YouTube   1134890  2987624        5.265            1.0           1       28754         50.7543  0.000005


In [16]:
# How do Facebook and Youtube Differ between Themselves and How can we solve this.
# This algorithm will take two different netwrk graphs and compare them to see which is more healthier and nicer.

class SocialNetwork:
    def __init__(self, name, graph):
        self.name  = name
        self.graph = graph
        self.stats = {}
        print(f"{self.name} Network is Ready with {graph.number_of_nodes():,} users and {graph.number_of_edges():,} connections")

    # We need to know what the heck our values would be like.
    # A network that isn't clustered or not would be different
    def compute_stats(self):
        degrees = [deg for _, deg in self.graph.degree()]

        self.stats = {
            "nodes"           : self.graph.number_of_nodes(),
            "edges"           : self.graph.number_of_edges(),
            "mean_degree"     : round(np.mean(degrees), 2),
            "median_degree"   : round(np.median(degrees), 2),
            "max_degree"      : int(np.max(degrees)),
            "min_degree"      : int(np.min(degrees)),
            "std_degree"      : round(np.std(degrees), 2),
            "density"         : round(nx.density(self.graph), 6),
            "avg_clustering"  : self._get_clustering(),
        }

        print(f"Total Stats Collected!")
        return self.stats
    

    # Check how tightly packed everyone is, how many friends do my friends have, etc.
    def _get_clustering(self):
        
        if self.graph.number_of_nodes() <= 50000:
            return round(nx.average_clustering(self.graph), 4)
        else:
            sample = list(self.graph.nodes())[:5000]
            return round(nx.average_clustering(self.graph, nodes=sample), 4)

    # Check who the people with the most connections are
    # They are influencers.
    # Basically the Superstars!
    def top_influencers(self, n=10):
        return sorted(self.graph.degree(), key=lambda x: x[1], reverse=True)[:n]

    # Check how connect the network is. How is the density.
    # Density here is a reference of how many connections are there compared to possible ones.
    def connectivity_score(self):
        if not self.stats:
            self.compute_stats()

        density_score    = self.stats["density"] * 10000  
        clustering_score = self.stats["avg_clustering"] * 100

        # We want to prioritize density more
        # 0.7 Density + 0.3 Clustering
        return round((density_score * 0.7) + (clustering_score * 0.3), 2)
    
    # Gives Reccomendations for Improving Health based on Stats
    def recommendations(self):
        if not self.stats:
            self.compute_stats()

        recs = []
        s = self.stats 

        # Clustering is high means good. Otherwise we need to improve.
        # Should be obvious but...
        if s["avg_clustering"] >= 0.3:
            recs.append(
                f"Communities are fairly nice! People have friends who are also friends."
            )
        else:
            recs.append(
                f"Commmunities are too weak. Not together enough.."
            )

        # How bad is the deviation of the connections. If some users have too much. Then it's bad.
        if s["std_degree"] > s["mean_degree"] * 2:
            recs.append(
                f"Too much connection inequality. Smallers users are too few."
            )
        else:
            recs.append(
                f"Decent Connections."
            )

        if s["density"] < 0.001:
            recs.append(
                f"Network users can't find each other easily!"
            )

        return recs

    # Final Print of all Found Stats
    def summary(self):
        if not self.stats:
            raise ValueError("This shouldn't be called until everything is done!")
        print(f"  {self.name.upper()} Summary of Values")
        for key, val in self.stats.items():
            label = key.replace("_", " ").title()
            print(f"  {label:<22}: {val:,}" if isinstance(val, int)
                  else f"  {label:<22}: {val}")
        print(f"\n  How good is the connection (0.7 Density + 0.3 Clustering) : {self.connectivity_score()} / 100")
        print(f"")

In [18]:
# Load the Class and Setup the Final Values

# Load the Datasets
fb = SocialNetwork("Facebook", facebook_graph)
yt = SocialNetwork("YouTube",  youtube_graph)

# Find the Values for Each Dataset's Nodes
fb.compute_stats()
yt.compute_stats()

# Get the Summary of Each Dataset
fb.summary()
yt.summary()

print("Top 10 Super Users with the Most Popularity!")
print("-" * 45)
for net in [fb, yt]:
    print(f"\n{net.name}:")
    for rank, (node, degree) in enumerate(net.top_influencers(10), 1):
        print(f"  #{rank:<3} User {node:<10} with {degree:,} connections")

# How connected are people in each?
fb_score = fb.connectivity_score()
yt_score = yt.connectivity_score()
winner   = "Facebook" if fb_score > yt_score else "YouTube"
print(f"0.3 Density + 0.7 Clustering Score:")
print(f"  Facebook has a score of {fb_score} / 100")
print(f"  YouTube  has a score of {yt_score} / 100")
print(f"  Social Network with Most Close Connections is {winner}")


Facebook Network is Ready with 4,039 users and 88,234 connections
YouTube Network is Ready with 1,134,890 users and 2,987,624 connections
Total Stats Collected!
Total Stats Collected!
  FACEBOOK Summary of Values
  Nodes                 : 4,039
  Edges                 : 88,234
  Mean Degree           : 43.69
  Median Degree         : 25.0
  Max Degree            : 1,045
  Min Degree            : 1
  Std Degree            : 52.41
  Density               : 0.01082
  Avg Clustering        : 0.6055

  How good is the connection (0.7 Density + 0.3 Clustering) : 93.91 / 100

  YOUTUBE Summary of Values
  Nodes                 : 1,134,890
  Edges                 : 2,987,624
  Mean Degree           : 5.27
  Median Degree         : 1.0
  Max Degree            : 28,754
  Min Degree            : 1
  Std Degree            : 50.75
  Density               : 5e-06
  Avg Clustering        : 0.1005

  How good is the connection (0.7 Density + 0.3 Clustering) : 3.05 / 100

Top 10 Super Users with the Mo

In [19]:
for net in [fb, yt]:
    print(f"\n{net.name}:")
    for rec in net.recommendations():
        print(f"  • {rec}")


Facebook:
  • Communities are fairly nice! People have friends who are also friends.
  • Decent Connections.

YouTube:
  • Commmunities are too weak. Not together enough..
  • Too much connection inequality. Smallers users are too few.
  • Network users can't find each other easily!
